In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [12]:
!pip install mlflow dagshub -q

import mlflow
import dagshub
from kaggle_secrets import UserSecretsClient
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("ML_assn1")   # your DagsHub token
dagshub_username = user_secrets.get_secret("username")  # your DagsHub username

os.environ["MLFLOW_TRACKING_USERNAME"] = dagshub_username
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

repo_owner = dagshub_username
repo_name = "ml_assn2"

mlflow.set_tracking_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")
mlflow.set_registry_uri(f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow")

dagshub.init(repo_name=repo_name, repo_owner=repo_owner, mlflow=True)

Initialized MLflow to track repo "lkuch23/ml_assn2"

Repository lkuch23/ml_assn2 initialized!

In [13]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
test = test_transaction.merge(test_identity, on='TransactionID', how='left')

# Cleaning

In [14]:
missing_ratio = train.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > 0.85].index
train.drop(cols_to_drop, axis=1, inplace=True)
test.drop(cols_to_drop, axis=1, inplace=True, errors='ignore')

In [15]:
for col in train.columns:
    if col == 'isFraud':
        continue
    if col in test.columns:
        if train[col].dtype == 'object':
            train[col] = train[col].fillna('missing')
            test[col] = test[col].fillna('missing')
        else:
            median = train[col].median()
            train[col] = train[col].fillna(median)
            test[col] = test[col].fillna(median)

# Feature Engineering

In [16]:
train['hour'] = (train['TransactionDT'] // 3600) % 24
test['hour'] = (test['TransactionDT'] // 3600) % 24
train['day_of_week'] = (train['TransactionDT'] // (3600*24)) % 7
test['day_of_week'] = (test['TransactionDT'] // (3600*24)) % 7
train['amt_log'] = np.log1p(train['TransactionAmt'])
test['amt_log'] = np.log1p(test['TransactionAmt'])
train['amt_cents'] = train['TransactionAmt'] - train['TransactionAmt'].astype(int)
test['amt_cents'] = test['TransactionAmt'] - test['TransactionAmt'].astype(int)
train['amt_per_hour'] = train['TransactionAmt'] / (train['hour'] + 1)
test['amt_per_hour'] = test['TransactionAmt'] / (test['hour'] + 1)

/tmp/ipykernel_57/496788992.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train['hour'] = (train['TransactionDT'] // 3600) % 24
/tmp/ipykernel_57/496788992.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test['hour'] = (test['TransactionDT'] // 3600) % 24
/tmp/ipykernel_57/496788992.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragm

In [17]:
for col in ['card1', 'card2', 'card3', 'card5']:
    if col in train.columns:
        grp = train.groupby(col)['TransactionAmt'].agg(['mean', 'std', 'count'])
        grp.columns = [f'{col}_amt_mean', f'{col}_amt_std', f'{col}_count']
        train = train.join(grp, on=col)
        test  = test.join(grp, on=col)

train.fillna(-999, inplace=True)
test.fillna(-999, inplace=True)

In [18]:
common_cols = list(set(train.columns) & set(test.columns))
cat_cols = [c for c in common_cols if train[c].dtype == 'object']
for col in cat_cols:
    le = LabelEncoder()
    le.fit(list(train[col].astype(str)) + list(test[col].astype(str)))
    train[col] = le.transform(train[col].astype(str))
    test[col] = le.transform(test[col].astype(str))

# Feature Selection

In [19]:
target = 'isFraud'
ID = 'TransactionID'
common_cols = set(train.columns) & set(test.columns)
features = sorted([c for c in common_cols if c not in [target, ID]])
X = train[features].apply(pd.to_numeric, errors='coerce').fillna(-999)
y = train[target]
X_test = test[features].apply(pd.to_numeric, errors='coerce').fillna(-999)

# Training

In [20]:
params = {
    'n_estimators': 300,
    'max_depth': 20,
    'min_samples_leaf': 5,
    'max_features': 'sqrt',
    'class_weight': 'balanced',
    'random_state': 42,
    'n_jobs': -1,
}

N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_aucs = []

mlflow.set_experiment("IEEE_RandomForest")

with mlflow.start_run():
    for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y)):
        X_trn, X_val = X.iloc[trn_idx], X.iloc[val_idx]
        y_trn, y_val = y.iloc[trn_idx], y.iloc[val_idx]
        model = RandomForestClassifier(**params)
        model.fit(X_trn, y_trn)
        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS
        fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
        fold_aucs.append(fold_auc)

    oof_auc = roc_auc_score(y, oof_preds)
    mlflow.log_metric("AUC", oof_auc)
    print(f"AUC: {oof_auc}")

AUC: 0.9367017305571698
🏃 View run bustling-loon-850 at: https://dagshub.com/lkuch23/ml_assn2.mlflow/#/experiments/2/runs/6ae28e2fe2154ab6bfda64ae475c92d7
🧪 View experiment at: https://dagshub.com/lkuch23/ml_assn2.mlflow/#/experiments/2
